In [1]:
import numpy as np
import pandas as pd
import sys

from pathlib import Path
from typing import Tuple

module_path = str(Path("..").resolve())
if module_path not in sys.path:
    sys.path.append(module_path)

import utils.data_io as data_io
import utils.vis as vis

from config_handlers.eval_handler import EvalHandler
from config_handlers.rk_handler import RKHandler

In [2]:
rk_hdl = RKHandler()
eval_hdl = EvalHandler()

model_list = rk_hdl.get_model_list()
""" Skipping MiniGPT4 as we have incomplete data. """
model_list.remove("minigpt4")
rk_hdl.set_curr_model(model_list[0])
ds_list = rk_hdl.get_ds_list()

all_ds = ds_list * len(model_list)
all_models = model_list * len(ds_list)
all_models.sort()

MC_DATASETS = ["seed", "vqav2"]
OPEN_DATASETS = ["llava-bench", "mmbench"]

CHARTS_DIR = Path("behaviour_charts")
data_io.make_dir(CHARTS_DIR)

[Errno 17] File exists: 'behaviour_charts'


In [3]:
def load_data(model_suffix: str = "mpnet", analysis_type=None) -> Tuple[dict, dict]:
    samples = {m: {ds: {} for ds in ds_list} for m in model_list}
    summaries = {m: {ds: {} for ds in ds_list} for m in model_list}
    if analysis_type:
        base_fp = Path("raw_stats", f"sim_{analysis_type}_{model_suffix}")
    else:
        base_fp = Path("raw_stats", f"sim_{model_suffix}")

    for model, ds in zip(all_models, all_ds):
        samples[model][ds] = data_io.load_json(
            base_fp.joinpath(model, ds, "samples.json")
        )
        summaries[model][ds] = data_io.load_json(
            base_fp.joinpath(model, ds, "summary.json")
        )
    return samples, summaries


def load_eval_responses():
    data = {m: {ds: {}} for m in model_list for ds in ds_list}
    for model, ds in zip(all_models, all_ds):
        eval_hdl.set_curr_model(model)
        eval_hdl.set_curr_ds(ds)
        data[model][ds] = data_io.load_json(Path("..", eval_hdl.get_eval_path()))
    return data

In [4]:
eval_data = load_eval_responses()

Check Accuracy (MC QA) and BERTScore F1 (open-ended QA)

In [5]:
eval_data_small = {m: {ds: 0.0 for ds in eval_data[m]} for m in eval_data}
for m in eval_data:
    print(f">> Eval for {m.upper()}")
    for ds in eval_data[m]:
        if "f1" in eval_data[m][ds]["measures"]:
            print(f" - BERTScore F1 on {ds}: {eval_data[m][ds]['measures']['f1']:.3f}")
        elif "accuracy" in eval_data[m][ds]["measures"]:
            print(
                f" - MC accuracy on {ds}: {eval_data[m][ds]['measures']['accuracy']:.3f}"
            )

>> Eval for INTERNVL2
 - MC accuracy on vqav2: 0.667
 - BERTScore F1 on llava-bench: 0.673
 - BERTScore F1 on mmbench: 0.796
 - MC accuracy on seed: 0.787
>> Eval for LLAVA-1.6
 - MC accuracy on vqav2: 0.585
 - BERTScore F1 on llava-bench: 0.657
 - BERTScore F1 on mmbench: 0.792
 - MC accuracy on seed: 0.667
>> Eval for SHAREGPT4V
 - MC accuracy on vqav2: 0.810
 - BERTScore F1 on llava-bench: 0.687
 - BERTScore F1 on mmbench: 0.705
 - MC accuracy on seed: 0.661
>> Eval for QWEN2_5_VL
 - MC accuracy on vqav2: 0.700
 - BERTScore F1 on llava-bench: 0.670
 - BERTScore F1 on mmbench: 0.743
 - MC accuracy on seed: 0.833


In [6]:
from matplotlib import pyplot as plt


def plot_bar(data, model, dataset, out_path):
    width = 0.5
    fig, ax = plt.subplots()
    fig.set_figwidth(20)
    # q_ids = list(data.keys())
    q_ids = list(np.arange(len(data.keys())))
    scores = list(data.values())

    p = ax.bar(q_ids, scores, width)

    # Add lines and counts for each threshold
    for y in np.arange(0.2, 1, 0.2):
        count = len([s for s in scores if s >= y])
        ax.text(-2, y, count, transform=ax.get_xaxis_transform())
        ax.axhline(y=y, color="r", linestyle=":")

    count = len([s for s in scores if s >= 0.7])
    ax.text(-2, 0.7, count, transform=ax.get_xaxis_transform())
    ax.axhline(y=0.7, color="g", linestyle="--")

    ax.set_title(f"BERTScore (F1) for {model.upper()} on {dataset.upper()}")
    # ax.legend(loc='upper right')
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()


# Prepare a bit the data
bert_scores = {model: {} for model in model_list}

for model in eval_data:
    for ds in OPEN_DATASETS:
        bert_scores[model][ds] = []
        for r in eval_data[model][ds]["ranges"]:
            bert_scores[model][ds].extend(eval_data[model][ds]["ranges"][r])

for model in bert_scores:
    for ds in bert_scores[model]:
        bert_scores[model][ds] = {
            question["question_id"]: question["measures"]["f1"]
            for question in bert_scores[model][ds]
        }

for model in bert_scores:
    for ds in bert_scores[model]:
        out_dir = Path("open_ended_perf")
        data_io.make_dir(out_dir)
        out_path = out_dir.joinpath(f"{model}_{ds}.pdf")
        plot_bar(bert_scores[model][ds], model, ds, out_path)

[Errno 17] File exists: 'open_ended_perf'
[Errno 17] File exists: 'open_ended_perf'
[Errno 17] File exists: 'open_ended_perf'
[Errno 17] File exists: 'open_ended_perf'
[Errno 17] File exists: 'open_ended_perf'
[Errno 17] File exists: 'open_ended_perf'
[Errno 17] File exists: 'open_ended_perf'
[Errno 17] File exists: 'open_ended_perf'


In [7]:
# For values and comparison with data obtained with nucleus sampling, check
# the "compare_decoding.ipynb" notebook.

model_suffix = "mpnet"  # granite
analysis_type = None
samples, summaries = load_data(model_suffix=model_suffix, analysis_type=analysis_type)

Check sample-level similarity values

In [8]:
for m, ds in zip(all_models, all_ds):
    print(f">> Results for {m.upper()} on {ds.upper()}")
    print(f"- Num. entries: {len(samples[m][ds])}")
    print(f"- Avg. SIM: {summaries[m][ds]['cosine_avg']:.3f}")
    # print(f"- Std. SIM: {summaries[m][ds]['cosine_std']}")
    print("-" * 30)

>> Results for INTERNVL2 on LLAVA-BENCH
- Num. entries: 58
- Avg. SIM: 0.564
------------------------------
>> Results for INTERNVL2 on MMBENCH
- Num. entries: 139
- Avg. SIM: 0.599
------------------------------
>> Results for INTERNVL2 on SEED
- Num. entries: 150
- Avg. SIM: 0.541
------------------------------
>> Results for INTERNVL2 on VQAV2
- Num. entries: 150
- Avg. SIM: 0.597
------------------------------
>> Results for LLAVA-1.6 on LLAVA-BENCH
- Num. entries: 44
- Avg. SIM: 0.520
------------------------------
>> Results for LLAVA-1.6 on MMBENCH
- Num. entries: 133
- Avg. SIM: 0.571
------------------------------
>> Results for LLAVA-1.6 on SEED
- Num. entries: 129
- Avg. SIM: 0.531
------------------------------
>> Results for LLAVA-1.6 on VQAV2
- Num. entries: 142
- Avg. SIM: 0.548
------------------------------
>> Results for QWEN2_5_VL on LLAVA-BENCH
- Num. entries: 56
- Avg. SIM: 0.391
------------------------------
>> Results for QWEN2_5_VL on MMBENCH
- Num. entries: 13

In [9]:
if analysis_type:
    fname = CHARTS_DIR.joinpath(f"distr_sim_{analysis_type}_{model_suffix}.pdf")
else:
    fname = CHARTS_DIR.joinpath(f"distr_sim_{model_suffix}.pdf")

vis.plot_similarity(samples, fname, measure_key="cosine")

Mapping behaviours types

In [10]:
df = pd.DataFrame()
for m, ds in zip(all_models, all_ds):
    for id_q in samples[m][ds]:
        cosine = (
            samples[m][ds][id_q]["cosine"]
            if samples[m][ds][id_q]["cosine"] > 0.0
            else 0.0
        )
        row = {
            "question_id": int(id_q),
            "model": m,
            "dataset": ds,
            "similarity": [cosine],
        }
        temp_df = pd.DataFrame.from_dict(row)
        df = pd.concat([df, temp_df], ignore_index=True)

In [11]:
counts_model = {m: {ds: len(samples[m][ds]) for ds in samples[m]} for m in samples}
counts_model

{'internvl2': {'llava-bench': 58, 'mmbench': 139, 'seed': 150, 'vqav2': 150},
 'llava-1.6': {'llava-bench': 44, 'mmbench': 133, 'seed': 129, 'vqav2': 142},
 'sharegpt4v': {'llava-bench': 49, 'mmbench': 135, 'seed': 109, 'vqav2': 21},
 'qwen2_5_vl': {'llava-bench': 56, 'mmbench': 139, 'seed': 150, 'vqav2': 150}}

In [12]:
for m, ds in zip(all_models, all_ds):
    print(f"{m} on {ds}: {len(df[(df['model'] == m) & (df['dataset'] == ds)])}")

internvl2 on llava-bench: 58
internvl2 on mmbench: 139
internvl2 on seed: 150
internvl2 on vqav2: 150
llava-1.6 on llava-bench: 44
llava-1.6 on mmbench: 133
llava-1.6 on seed: 129
llava-1.6 on vqav2: 142
qwen2_5_vl on llava-bench: 56
qwen2_5_vl on mmbench: 139
qwen2_5_vl on seed: 150
qwen2_5_vl on vqav2: 150
sharegpt4v on llava-bench: 49
sharegpt4v on mmbench: 135
sharegpt4v on seed: 109
sharegpt4v on vqav2: 21


In [13]:
df.head()

,question_id,model,dataset,similarity
0,0,internvl2,llava-bench,0.662013
1,1,internvl2,llava-bench,0.716396
2,2,internvl2,llava-bench,0.588038
3,3,internvl2,llava-bench,0.457952
4,4,internvl2,llava-bench,0.423745


In [14]:
# Preparing the lists of question_ids, organised in buckets
flat_ids = {model: {ds: {} for ds in eval_data[model]} for model in eval_data}

for model, ds in zip(all_models, all_ds):
    if ds in OPEN_DATASETS:
        # Parse ranges
        ranges = list(eval_data[model][ds]["ranges"].keys())
        for r in ranges:
            flat_ids[model][ds][r] = [
                int(q["question_id"]) for q in eval_data[model][ds]["ranges"][r]
            ]
    elif ds in MC_DATASETS:
        # Parse correct_resp and wrong_resp
        flat_ids[model][ds]["correct_resp"] = [
            int(q["question_id"]) for q in eval_data[model][ds]["correct_resp"]
        ]
        flat_ids[model][ds]["wrong_resp"] = [
            int(q["question_id"]) for q in eval_data[model][ds]["correct_resp"]
        ]
    else:
        print("Error.")
    # Parse invalid_resp
    flat_ids[model][ds]["invalid_resp"] = [
        int(q["question_id"]) for q in eval_data[model][ds]["invalid_resp"]
    ]
    print(f"Processed {model} on {ds}.")

Processed internvl2 on llava-bench.
Processed internvl2 on mmbench.
Processed internvl2 on seed.
Processed internvl2 on vqav2.
Processed llava-1.6 on llava-bench.
Processed llava-1.6 on mmbench.
Processed llava-1.6 on seed.
Processed llava-1.6 on vqav2.
Processed qwen2_5_vl on llava-bench.
Processed qwen2_5_vl on mmbench.
Processed qwen2_5_vl on seed.
Processed qwen2_5_vl on vqav2.
Processed sharegpt4v on llava-bench.
Processed sharegpt4v on mmbench.
Processed sharegpt4v on seed.
Processed sharegpt4v on vqav2.


In [15]:
def get_outcome(x, flat_ids):
    for k, v in flat_ids[x["model"]][x["dataset"]].items():
        if int(x["question_id"]) in v:
            # Fix outcome for MC questions
            if x["dataset"] in MC_DATASETS:
                return "correct" if k == "correct_resp" else "wrong"
            else:
                return k
    return None

In [16]:
df["outcome"] = df.apply(lambda x: get_outcome(x, flat_ids), axis=1)

In [17]:
df.head()

,question_id,model,dataset,similarity,outcome
0,0,internvl2,llava-bench,0.662013,60_80
1,1,internvl2,llava-bench,0.716396,60_80
2,2,internvl2,llava-bench,0.588038,60_80
3,3,internvl2,llava-bench,0.457952,40_60
4,4,internvl2,llava-bench,0.423745,40_60


In [18]:
df[df["dataset"] == "seed"].head()

,question_id,model,dataset,similarity,outcome
197,3151,internvl2,seed,0.578140,correct
198,1656,internvl2,seed,0.448091,correct
199,154,internvl2,seed,0.711095,correct
200,3057,internvl2,seed,0.765221,correct
201,2173,internvl2,seed,0.429385,correct


In [19]:
df["question_type"] = df.apply(
    lambda x: "open" if x["dataset"] in OPEN_DATASETS else "closed", axis=1
)
df["open_res"] = df.apply(
    lambda x: x["outcome"] if x["question_type"] == "open" else None, axis=1
)
df["closed_res"] = df.apply(
    lambda x: x["outcome"] if x["question_type"] == "closed" else None, axis=1
)

In [20]:
df.head()

,question_id,model,dataset,similarity,outcome,question_type,open_res,closed_res
0,0,internvl2,llava-bench,0.662013,60_80,open,60_80,None
1,1,internvl2,llava-bench,0.716396,60_80,open,60_80,None
2,2,internvl2,llava-bench,0.588038,60_80,open,60_80,None
3,3,internvl2,llava-bench,0.457952,40_60,open,40_60,None
4,4,internvl2,llava-bench,0.423745,40_60,open,40_60,None


In [21]:
df[df["dataset"] == "seed"].head()

,question_id,model,dataset,similarity,outcome,question_type,open_res,closed_res
197,3151,internvl2,seed,0.578140,correct,closed,None,correct
198,1656,internvl2,seed,0.448091,correct,closed,None,correct
199,154,internvl2,seed,0.711095,correct,closed,None,correct
200,3057,internvl2,seed,0.765221,correct,closed,None,correct
201,2173,internvl2,seed,0.429385,correct,closed,None,correct


Create ranges for cosine similarity

In [22]:
# Simple equal 3-way split
prec = 6
n_split = 3
step = round(1 / n_split, prec)
simple_split = {
    idx + 1: {"start": round(val.item(), prec), "end": round(val.item() + step, prec)}
    for idx, val in enumerate(np.linspace(0, 1, n_split + 1))
    if idx < n_split
}

# Swap ranges for types 1 and 3
temp = simple_split[1]
simple_split[1] = simple_split[3]
simple_split[3] = temp

print(simple_split)

{1: {'start': 0.666667, 'end': 1.0}, 2: {'start': 0.333333, 'end': 0.666666}, 3: {'start': 0.0, 'end': 0.333333}}


Compute counts

In [23]:
df.head()

,question_id,model,dataset,similarity,outcome,question_type,open_res,closed_res
0,0,internvl2,llava-bench,0.662013,60_80,open,60_80,None
1,1,internvl2,llava-bench,0.716396,60_80,open,60_80,None
2,2,internvl2,llava-bench,0.588038,60_80,open,60_80,None
3,3,internvl2,llava-bench,0.457952,40_60,open,40_60,None
4,4,internvl2,llava-bench,0.423745,40_60,open,40_60,None


In [24]:
def get_counts(df, ranges, aggregation="plot"):
    if aggregation == "model":
        counts = {m: {ds: {} for ds in ds_list} for m in model_list}
    elif aggregation == "type":
        counts = {
            idx: {m: {ds: 0 for ds in ds_list} for m in model_list} for idx in ranges
        }
    elif aggregation == "plot":
        counts = {idx: {ds: [] for ds in ds_list} for idx in ranges}
    else:
        print("aggregation not recognised")
        return

    for idx, r in ranges.items():
        for m, ds in zip(all_models, all_ds):
            # Get counts
            count = len(
                df[
                    (df["dataset"] == ds)
                    & (df["model"] == m)
                    & (round(df["similarity"], prec) >= r["start"])
                    & (round(df["similarity"], prec) <= r["end"])
                ]
            )

            if aggregation == "model":
                counts[m][ds][idx] = count
            elif aggregation == "type":
                counts[idx][m][ds] = count
            elif aggregation == "plot":
                counts[idx][ds].append(count)
            else:
                print("aggregation not recognised")
                return

    # if aggregation == "plot":
    #     # Aligning the data for plotting
    #     # Swap: 0 (Type 3 failures) <-> 2 (Type 1 failures)
    #     # Good: 1 (Type 2 failures)
    #     temp = counts[0]
    #     counts[0] = counts[2]
    #     counts[2] = temp

    return counts

In [25]:
counts_model = get_counts(df, simple_split, aggregation="model")
counts_type = get_counts(df, simple_split, aggregation="type")
counts_plot = get_counts(df, simple_split, aggregation="plot")

In [26]:
counts_plot

{1: {'llava-bench': [12, 11, 7, 10],
  'mmbench': [50, 40, 56, 44],
  'seed': [39, 28, 15, 17],
  'vqav2': [51, 29, 25, 5]},
 2: {'llava-bench': [44, 25, 24, 33],
  'mmbench': [81, 80, 69, 81],
  'seed': [93, 81, 87, 60],
  'vqav2': [92, 100, 92, 15]},
 3: {'llava-bench': [2, 8, 25, 6],
  'mmbench': [8, 13, 14, 10],
  'seed': [18, 20, 48, 32],
  'vqav2': [7, 13, 33, 1]}}

In [27]:
import itertools


def check_counts(counts):
    s = 0
    for bucket, data in counts.items():
        bucket_total = sum(list(itertools.chain.from_iterable(data.values())))
        print(f"{bucket}: {bucket_total}")
        s += bucket_total
    print(f">> Total: {s}")


check_counts(counts_plot)

1: 439
2: 1057
3: 258
>> Total: 1754


In [28]:
# Need to save for each model and dataset the ids that are "considered"
ids_saved = {m: {ds: [] for ds in ds_list} for m in model_list}

for idx, r in simple_split.items():
    for m, ds in zip(all_models, all_ds):
        subdf = df[(df["dataset"] == ds) & (df["model"] == m)]
        for _, row in subdf.iterrows():
            if (
                round(row["similarity"], prec) >= r["start"]
                and round(row["similarity"], prec) <= r["end"]
            ):
                ids_saved[m][ds].append(row["question_id"])

# Loop over question_ids in df and look for the one that has not been saved
problems = []
for m, ds in zip(all_models, all_ds):
    subdf = df[(df["dataset"] == ds) & (df["model"] == m)]
    for _, row in subdf.iterrows():
        if row["question_id"] not in ids_saved[m][ds]:
            problems.append(row)
print(len(problems))

0


In [29]:
if analysis_type:
    fname = CHARTS_DIR.joinpath(f"sample_types_{analysis_type}_{model_suffix}.pdf")
else:
    fname = CHARTS_DIR.joinpath(f"sample_types_{model_suffix}.pdf")
vis.plot_behaviour_counts(counts_plot, fname, to_skip="minigpt4")

Skipping data for minigpt4


/Users/lorenzocorti/Developer/mllm_diagnosis/utils/vis.py:458: UserWarning: The figure layout has changed to tight
  plt.tight_layout()


In [30]:
# Ad-hoc splits
# strict_split = {
#     0: {"start": 0.0, "end": 0.199999},
#     1: {"start": 0.200000, "end": 0.799999},
#     2: {"start": 0.800, "end": 1.0},
# }
strict_split = {
    1: {"start": 0.700000, "end": 1.0},
    2: {"start": 0.333333, "end": 0.699999},
    3: {"start": 0.0, "end": 0.333332},
}
print(strict_split)

{1: {'start': 0.7, 'end': 1.0}, 2: {'start': 0.333333, 'end': 0.699999}, 3: {'start': 0.0, 'end': 0.333332}}


In [31]:
counts_model = get_counts(df, strict_split, aggregation="model")
counts_type = get_counts(df, strict_split, aggregation="type")
counts_plot = get_counts(df, strict_split, aggregation="plot")

In [32]:
check_counts(counts_plot)

1: 314
2: 1182
3: 258
>> Total: 1754


In [ ]:
if analysis_type:
    fname = CHARTS_DIR.joinpath(
        f"sample_types_strict_{analysis_type}_{model_suffix}.pdf"
    )
else:
    fname = CHARTS_DIR.joinpath(f"sample_types_strict_{model_suffix}.pdf")
vis.plot_behaviour_counts(counts_plot, fname, to_skip="minigpt4")

Skipping data for minigpt4


/Users/lorenzocorti/Developer/mllm_diagnosis/utils/vis.py:458: UserWarning: The figure layout has changed to tight
  plt.tight_layout()
